# 3. HMM State Discovery (BE & SC)

**Purpose**: Fit Hidden Markov Models to session-level summary statistics from
both BE and SC synthetic data. Discover behavioural states in a model-free manner
and validate that HMM states correspond to true learning phases.

**Key question**: Do HMM state assignments agree for data generated by different
models? If the HMM finds the same states regardless of generative model, the
state definitions are robust.


In [ ]:
import sys
import os

sys.path.append(os.path.abspath(".."))

%matplotlib inline
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings
warnings.filterwarnings('ignore')

try:
    from hmmlearn.hmm import GaussianHMM
    HMM_AVAILABLE = True
except ImportError:
    print("hmmlearn not installed. Install with: pip install hmmlearn")
    HMM_AVAILABLE = False

from Models.BE_core import BEParams
from Models.SC_core import SCParams
from Data.structures import (
    generate_synthetic_animal,
    param_trajectory_naive_to_expert,
    sc_param_trajectory_naive_to_expert,
)
from Analysis.summary_stats import (
    compute_summary_stats, get_stat_names_expanded, list_available_stats,
)
from Analysis.session_features import build_feature_matrix_multi

plt.rcParams['figure.dpi'] = 100

## 1. Generate Synthetic Data (Both Models)

In [ ]:
# --- Configuration ---
N_ANIMALS = 8
N_SESSIONS = 30
TRIALS_PER = 350
BURN_IN = 100
SEED = 42

# Stats for HMM features
HMM_STATS = [
    'accuracy', 'recency', 'pse', 'slope',
    'stimulus_sensitivity', 'win_stay_rate',
    'choice_entropy', 'perseveration',
    'hard_accuracy',
]

rng_master = np.random.default_rng(SEED)
expanded_names = get_stat_names_expanded(HMM_STATS)
print(f"HMM features: {expanded_names}")

In [ ]:
# --- Generate BE and SC animals with varied trajectories ---

def make_animals(model_type, n_animals, seed_base):
    animals = []
    eta_trajs = []
    for i in range(n_animals):
        if model_type == 'BE':
            # Vary eta_start and eta_end across animals
            eta_s = 0.08 + rng_master.uniform(-0.02, 0.04)
            eta_e = 0.001 + rng_master.uniform(0, 0.005)
            params = param_trajectory_naive_to_expert(
                N_SESSIONS, eta_start=eta_s, eta_end=eta_e)
            eta_traj = params['eta_learning']
        else:
            gamma_s = 0.83 + rng_master.uniform(-0.02, 0.04)
            gamma_e = 0.975 + rng_master.uniform(0, 0.005)
            params = sc_param_trajectory_naive_to_expert(
                N_SESSIONS, gamma_start=gamma_s, gamma_end=gamma_e)
            eta_traj = [1 - g for g in params['gamma']]  # effective learning rate

        animal, gt = generate_synthetic_animal(
            animal_id=f'{model_type}_{i:02d}',
            n_sessions=N_SESSIONS,
            trials_per_session=TRIALS_PER,
            true_params=params,
            burn_in=BURN_IN,
            seed=seed_base + i,
            model_type=model_type,
        )
        animals.append(animal)
        eta_trajs.append(eta_traj)
    return animals, eta_trajs

be_animals, be_etas = make_animals('BE', N_ANIMALS, 100)
sc_animals, sc_etas = make_animals('SC', N_ANIMALS, 200)
print(f"Generated {len(be_animals)} BE + {len(sc_animals)} SC animals")

In [ ]:
# --- Build feature matrices ---
ALL_STAT_NAMES = list_available_stats()

df_be = build_feature_matrix_multi(be_animals, stage=None,
                                    stat_names=ALL_STAT_NAMES, compute_deltas=False)
df_be['model'] = 'BE'

df_sc = build_feature_matrix_multi(sc_animals, stage=None,
                                    stat_names=ALL_STAT_NAMES, compute_deltas=False)
df_sc['model'] = 'SC'

# Add true eta trajectories
for df, etas, prefix in [(df_be, be_etas, 'BE'), (df_sc, sc_etas, 'SC')]:
    true_eta = []
    for _, row in df.iterrows():
        aid = row['animal_id']
        idx = int(aid.split('_')[1])
        sess = int(row['session_idx'])
        true_eta.append(etas[idx][sess] if sess < len(etas[idx]) else np.nan)
    df['true_eta'] = true_eta

df_all = pd.concat([df_be, df_sc], ignore_index=True)
print(f"Combined: {len(df_all)} sessions, {len(df_all.columns)} columns")

## 2. Preprocessing: Z-score and PCA

In [ ]:
# --- Select features and z-score ---
feat_cols = [c for c in expanded_names if c in df_all.columns]
missing = [c for c in expanded_names if c not in df_all.columns]
if missing:
    print(f"Missing features (skipping): {missing}")

X_raw = df_all[feat_cols].values
valid_mask = ~np.any(np.isnan(X_raw), axis=1)
X_clean = X_raw[valid_mask]

scaler = StandardScaler()
X_z = scaler.fit_transform(X_clean)
print(f"Feature matrix: {X_z.shape} (after dropping {(~valid_mask).sum()} NaN rows)")

# --- PCA ---
pca = PCA()
X_pca_full = pca.fit_transform(X_z)
cumvar = np.cumsum(pca.explained_variance_ratio_)
n_pcs_90 = np.searchsorted(cumvar, 0.90) + 1
N_PCS = max(3, n_pcs_90)
X_pca = X_pca_full[:, :N_PCS]

print(f"Using {N_PCS} PCs (explains {cumvar[N_PCS-1]*100:.1f}% variance)")

fig, ax = plt.subplots(figsize=(8, 3))
ax.bar(range(1, len(cumvar)+1), pca.explained_variance_ratio_, color='steelblue', alpha=0.6)
ax.plot(range(1, len(cumvar)+1), cumvar, 'ko-', ms=4)
ax.axhline(0.9, ls='--', color='red', lw=0.8)
ax.axvline(N_PCS, ls='--', color='green', lw=0.8)
ax.set_xlabel('PC'); ax.set_ylabel('Variance explained')
ax.set_title('PCA Scree Plot')
plt.tight_layout()
plt.show()

## 3. Fit HMMs with Model Selection

In [ ]:
if HMM_AVAILABLE:
    # --- Prepare sequence lengths per animal ---
    df_valid = df_all.iloc[np.where(valid_mask)[0]].copy()
    df_valid['_pca_idx'] = range(len(X_pca))

    # Build per-animal lengths
    lengths_all = []
    X_ordered = []
    for aid in df_valid['animal_id'].unique():
        sub = df_valid[df_valid['animal_id'] == aid].sort_values('session_idx')
        idxs = sub['_pca_idx'].values
        X_ordered.append(X_pca[idxs])
        lengths_all.append(len(idxs))

    X_concat = np.vstack(X_ordered)
    lengths_arr = np.array(lengths_all)
    print(f"HMM input: {X_concat.shape}, {len(lengths_arr)} sequences")

    # --- Fit HMMs with 2-6 states ---
    K_range = range(2, 7)
    results = {}

    for K in K_range:
        best_ll = -np.inf
        best_model = None
        for init in range(10):
            try:
                model = GaussianHMM(n_components=K, covariance_type='full',
                                     n_iter=200, random_state=SEED + init)
                model.fit(X_concat, lengths_arr)
                ll = model.score(X_concat, lengths_arr)
                if ll > best_ll:
                    best_ll = ll
                    best_model = model
            except Exception:
                pass

        if best_model is not None:
            n_params = K * N_PCS + K * N_PCS * (N_PCS + 1) // 2 + K * K
            bic = -2 * best_ll + n_params * np.log(len(X_concat))
            aic = -2 * best_ll + 2 * n_params
            states = best_model.predict(X_concat, lengths_arr)
            results[K] = {'model': best_model, 'bic': bic, 'aic': aic,
                           'll': best_ll, 'states': states}
            print(f"  K={K}: BIC={bic:.0f}, AIC={aic:.0f}, LL={best_ll:.0f}")

    best_K = min(results, key=lambda k: results[k]['bic'])
    print(f"\nBest K by BIC: {best_K}")
else:
    print("hmmlearn not available — skipping HMM fitting.")

In [ ]:
# --- Model selection plot ---
if HMM_AVAILABLE and results:
    Ks = sorted(results.keys())
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

    ax1.plot(Ks, [results[k]['bic'] for k in Ks], 'o-', color='steelblue', label='BIC')
    ax1.plot(Ks, [results[k]['aic'] for k in Ks], 's-', color='darkorange', label='AIC')
    ax1.axvline(best_K, ls='--', color='red', alpha=0.5)
    ax1.set_xlabel('Number of states')
    ax1.set_ylabel('Information criterion')
    ax1.legend()
    ax1.set_title('Model Selection')

    ax2.plot(Ks, [results[k]['ll'] for k in Ks], 'o-', color='green')
    ax2.set_xlabel('Number of states')
    ax2.set_ylabel('Log-likelihood')
    ax2.set_title('Log-Likelihood')

    plt.tight_layout()
    plt.show()

## 4. Visualise State Sequences — BE vs SC

In [ ]:
if HMM_AVAILABLE and results:
    best = results[best_K]
    states_all = best['states']
    df_valid['hmm_state'] = states_all

    fig, axes = plt.subplots(N_ANIMALS, 2, figsize=(16, 2 * N_ANIMALS),
                              sharex=True, sharey=True)
    fig.suptitle(f'HMM States (K={best_K}): BE (left) vs SC (right)',
                 fontsize=13, fontweight='bold')

    cmap = plt.cm.Set2
    state_colours = {s: cmap(s) for s in range(best_K)}

    for row in range(N_ANIMALS):
        for col, model_name in enumerate(['BE', 'SC']):
            ax = axes[row, col]
            aid = f'{model_name}_{row:02d}'
            sub = df_valid[df_valid['animal_id'] == aid].sort_values('session_idx')

            if len(sub) == 0:
                ax.set_visible(False)
                continue

            # State as coloured bars
            for _, r in sub.iterrows():
                ax.axvspan(r['session_idx'] - 0.4, r['session_idx'] + 0.4,
                           color=state_colours[r['hmm_state']], alpha=0.6)

            # True eta overlay
            ax2 = ax.twinx()
            ax2.plot(sub['session_idx'], sub['true_eta'], 'k-', lw=1.5, alpha=0.8)
            ax2.set_ylabel('η', fontsize=8)

            if col == 0:
                ax.set_ylabel(f'Animal {row}', fontsize=8)
            if row == 0:
                ax.set_title(f'{model_name}', fontsize=10)

    axes[-1, 0].set_xlabel('Session')
    axes[-1, 1].set_xlabel('Session')

    # Legend
    from matplotlib.patches import Patch
    legend_elements = [Patch(facecolor=state_colours[s], label=f'State {s}')
                        for s in range(best_K)]
    fig.legend(handles=legend_elements, loc='upper right', fontsize=9)

    plt.tight_layout()
    plt.show()

## 5. Validation: HMM States vs True η

In [ ]:
if HMM_AVAILABLE and results:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

    for ax, model_name, colour in [(ax1, 'BE', 'steelblue'), (ax2, 'SC', 'darkorange')]:
        sub = df_valid[df_valid['model'] == model_name]
        for state in range(best_K):
            state_etas = sub[sub['hmm_state'] == state]['true_eta'].dropna()
            if len(state_etas) > 0:
                vp = ax.violinplot(state_etas, positions=[state], widths=0.7,
                                    showmedians=True)
                for body in vp.get('bodies', []):
                    body.set_facecolor(state_colours[state])
                    body.set_alpha(0.6)

        ax.set_xlabel('HMM State')
        ax.set_ylabel('True η (effective learning rate)')
        ax.set_title(f'{model_name}: η distribution per HMM state')
        ax.set_xticks(range(best_K))

    plt.tight_layout()
    plt.show()

    # Print summary
    print("Mean η per HMM state:")
    for model_name in ['BE', 'SC']:
        sub = df_valid[df_valid['model'] == model_name]
        print(f"  {model_name}:")
        for state in range(best_K):
            eta_vals = sub[sub['hmm_state'] == state]['true_eta'].dropna()
            if len(eta_vals) > 0:
                print(f"    State {state}: η = {eta_vals.mean():.4f} ± {eta_vals.std():.4f} "
                      f"(n={len(eta_vals)})")

## 6. Cross-Model Consistency

Key test: do sessions assigned to the same HMM state from BE and SC data
have similar summary stat profiles? If yes, the HMM state definitions are
robust to the generative model.

In [ ]:
if HMM_AVAILABLE and results:
    # Compare stat profiles per state, BE vs SC
    fig, axes = plt.subplots(1, best_K, figsize=(5 * best_K, 5))
    if best_K == 1:
        axes = [axes]

    for state, ax in enumerate(axes):
        be_state = df_valid[(df_valid['model'] == 'BE') & (df_valid['hmm_state'] == state)]
        sc_state = df_valid[(df_valid['model'] == 'SC') & (df_valid['hmm_state'] == state)]

        stats_to_compare = [s for s in feat_cols if s in be_state.columns][:10]
        x = np.arange(len(stats_to_compare))

        be_means = [be_state[s].mean() for s in stats_to_compare]
        sc_means = [sc_state[s].mean() for s in stats_to_compare]

        ax.barh(x - 0.15, be_means, 0.3, color='steelblue', alpha=0.7, label='BE')
        ax.barh(x + 0.15, sc_means, 0.3, color='darkorange', alpha=0.7, label='SC')
        ax.set_yticks(x)
        ax.set_yticklabels(stats_to_compare, fontsize=8)
        ax.set_title(f'State {state} (n_BE={len(be_state)}, n_SC={len(sc_state)})')
        ax.legend(fontsize=8)

    plt.suptitle('Stat Profiles per HMM State: BE vs SC', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()

## Summary

**Convergence**: If the same HMM states emerge from both BE and SC data with
similar η distributions, the state definitions are model-agnostic — they capture
genuine behavioural transitions rather than model-specific artefacts.

**Divergence**: If BE and SC produce different state structures, this reveals
where the models predict different behavioural signatures, which is itself informative
for model comparison.

**Next**: Apply the fitted HMM to real data (same z-scoring + PCA transform) to
classify real sessions and trigger optogenetic/imaging experiments.
